# Module 6: Connecting tools through the platform

> **Prototype notebook.** It walks this module's steps with the evidence kept on the page and links
> straight to the right place in MLflow and the OpenShift console. The wording is a working draft;
> the module's published text is the lab author's.

In [ ]:
import labnb as lab
%cd ../agent

## Start from module 6

This puts the agent in the state module 6 starts from. Any edits you made are copied to
`agent/.reset-backup/` first, so nothing you wrote is lost.

In [ ]:
!python reset.py 6

## 1. How the agent gets its tools today

Every tool carries its own address and credential, written into the agent's configuration:

In [ ]:
import json
tools = lab.config()["tools"]
print("gateway_url:", repr(tools.get("gateway_url")))
print(json.dumps({k: v for k, v in tools.items() if k != "gateway_url"}, indent=2))

Run module 3's question on the agent's own tools, to compare with later:

In [ ]:
!python agent.py --q m3
lab.evidence(columns=("turns", "ended by", "tools run"))

Adding a tool today means editing the agent, rebuilding its image, redeploying it, and doing that again
in every environment it runs in. Tool changes are application releases.

```mermaid
flowchart TB
  subgraph Today["Today"]
    A1["agent"] --> T1["tools.py: find_part, check_stock, search_bulletins"]
  end
  subgraph Platform["With MCP Gateway"]
    A2["agent"] -- "one gateway_url" --> G["MCP Gateway (Tech Preview)"]
    G --> S1["The lab's tool server"]
    G -. "registered by the platform team" .-> S2["estimate_delivery"]
  end
  Today ~~~ Platform
```

## 2. Point the agent at the gateway

> **Not available on this environment yet.** This module needs the lab's MCP Gateway, with the
> lab's tools served behind it and a way to register `estimate_delivery`. On the dev cluster the
> gateway's tool server is still a placeholder. The cells below say so instead of failing. When the
> environment provides it, its address is in `LAB_GATEWAY_URL`.

In [ ]:
import os
GATEWAY = os.environ.get("LAB_GATEWAY_URL")
print(GATEWAY or "LAB_GATEWAY_URL isn't set: this environment has no lab MCP Gateway yet.")

With the gateway available: open [`config.toml`](../agent/config.toml), set `gateway_url` under
`[tools]` to the address above, and save. That's the only change to the agent.

## 3. Nothing changes about what the agent does

In [ ]:
if lab.config()["tools"].get("gateway_url"):
    lab.sh("python agent.py --q m3")
    lab.evidence(columns=("turns", "ended by", "tools run"), last=2)
else:
    print("gateway_url is empty, so the agent still uses its own tools.")

Same turns, same results; the tool names now carry the gateway's `kestrel_` prefix. The agent asked
the gateway for its tool list at the start of the run instead of carrying one. To see that list:

In [ ]:
if lab.config()["tools"].get("gateway_url"):
    lab.tool_list()
else:
    print("Set gateway_url first.")

## 4. A new tool, without touching the agent

Registering `estimate_delivery` at the gateway is a platform operation: nothing in the agent changes,
and it isn't rebuilt or restarted. Once it's registered, ask:

In [ ]:
if lab.config()["tools"].get("gateway_url"):
    lab.sh("python agent.py --q m6")
    lab.tool_results()
    lab.trace_link()
else:
    print("Set gateway_url first.")

**This is the module's success signal:** the agent discovered and used `estimate_delivery`, a tool it
didn't have a moment ago, and answered "1 day". The tool catalogue is now a property of the platform,
versioned and changed independently of the agent's releases.